# MiWay GTFS Route Efficiency Project — Notebook 1

## Goal of this notebook
This notebook sets up the foundation for the MiWay route efficiency project. It:

1. Loads the MiWay GTFS scheduled transit feed from `google_transit.zip`
2. Inspects the key GTFS tables
3. Confirms the feed validity period
4. Selects a representative weekday: **Tuesday, May 5, 2026**
5. Filters the GTFS trips to only those operating on that date
6. Produces a clean starting table of active routes and trips for later efficiency analysis

Later notebooks will build on this to calculate:
- route length,
- route directness,
- scheduled travel speed,
- stop density,
- service frequency/headways,
- and a final route efficiency score.

## 1. Import libraries

We use only common Python packages here: `pandas`, `zipfile`, and `pathlib`.

In [ ]:
import zipfile
from pathlib import Path
from datetime import datetime

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

## 2. Locate the GTFS ZIP file

This code checks a few common file locations. Keep `google_transit.zip` either:

- in the same folder as this notebook,
- inside a `data/` folder,
- or inside `../data/` if your notebook is in a `notebooks/` folder.

If none of those locations work, edit the `candidate_paths` list.

In [ ]:
candidate_paths = [
    Path('google_transit.zip'),
    Path('data/google_transit.zip'),
    Path('../data/google_transit.zip'),
]

ZIP_PATH = next((path for path in candidate_paths if path.exists()), None)

if ZIP_PATH is None:
    raise FileNotFoundError(
        'Could not find google_transit.zip. Place it beside this notebook or inside a data/ folder.'
    )

print(f'Using GTFS ZIP file: {ZIP_PATH.resolve()}')

## 3. Inspect the ZIP contents

A GTFS feed is a ZIP file containing text tables. For this project, the most important ones are:

- `routes.txt`
- `trips.txt`
- `stop_times.txt`
- `stops.txt`
- `shapes.txt`
- `calendar_dates.txt`
- `feed_info.txt`

In [ ]:
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zip_contents = zf.namelist()

zip_contents

## 4. Load GTFS tables into pandas

These are the core tables we will use throughout the project.

In [ ]:
def read_gtfs_table(zip_path: Path, filename: str) -> pd.DataFrame:
    """Read one GTFS text file directly from a ZIP archive into a DataFrame."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        if filename not in zf.namelist():
            raise FileNotFoundError(f'{filename} is not present in {zip_path.name}')
        with zf.open(filename) as file:
            return pd.read_csv(file)


routes = read_gtfs_table(ZIP_PATH, 'routes.txt')
trips = read_gtfs_table(ZIP_PATH, 'trips.txt')
stop_times = read_gtfs_table(ZIP_PATH, 'stop_times.txt')
stops = read_gtfs_table(ZIP_PATH, 'stops.txt')
shapes = read_gtfs_table(ZIP_PATH, 'shapes.txt')
calendar_dates = read_gtfs_table(ZIP_PATH, 'calendar_dates.txt')
feed_info = read_gtfs_table(ZIP_PATH, 'feed_info.txt')

print('Loaded GTFS tables successfully.')

## 5. Basic dataset inventory

Before doing any analysis, confirm the size of each table.

In [ ]:
gtfs_inventory = pd.DataFrame({
    'table': [
        'routes', 'trips', 'stop_times', 'stops',
        'shapes', 'calendar_dates', 'feed_info'
    ],
    'rows': [
        len(routes), len(trips), len(stop_times), len(stops),
        len(shapes), len(calendar_dates), len(feed_info)
    ],
    'columns': [
        routes.shape[1], trips.shape[1], stop_times.shape[1], stops.shape[1],
        shapes.shape[1], calendar_dates.shape[1], feed_info.shape[1]
    ]
})

gtfs_inventory

## 6. Preview the main tables

These quick previews help us understand the field names before doing joins.

In [ ]:
routes.head()

In [ ]:
trips.head()

In [ ]:
stop_times.head()

In [ ]:
stops.head()

In [ ]:
shapes.head()

## 7. Confirm the feed validity dates

The feed info table tells us the date range covered by this GTFS download.

In [ ]:
feed_info

In [ ]:
feed_start = pd.to_datetime(str(feed_info.loc[0, 'feed_start_date']), format='%Y%m%d')
feed_end = pd.to_datetime(str(feed_info.loc[0, 'feed_end_date']), format='%Y%m%d')

print(f'Feed start date: {feed_start:%B %d, %Y}')
print(f'Feed end date:   {feed_end:%B %d, %Y}')

## 8. Choose a representative analysis date

For the first version of the project, we use:

> **Tuesday, May 5, 2026**

This gives us a normal weekday service day and avoids mixing weekday and weekend schedules.

In [ ]:
ANALYSIS_DATE = pd.Timestamp('2026-05-05')
ANALYSIS_DATE_INT = int(ANALYSIS_DATE.strftime('%Y%m%d'))

print(f'Analysis date: {ANALYSIS_DATE:%A, %B %d, %Y}')
print(f'GTFS date code: {ANALYSIS_DATE_INT}')

## 9. Find service IDs active on the analysis date

This feed uses `calendar_dates.txt` rather than a regular `calendar.txt` table. In GTFS:

- `exception_type = 1` means service is **added / operating** on that date.
- `exception_type = 2` means service is **removed** on that date.

For May 5, 2026, we keep the service IDs marked as operating.

In [ ]:
active_services = calendar_dates.loc[
    (calendar_dates['date'] == ANALYSIS_DATE_INT) &
    (calendar_dates['exception_type'] == 1),
    'service_id'
].drop_duplicates()

print(f'Active service IDs on {ANALYSIS_DATE:%Y-%m-%d}: {len(active_services)}')
active_services.tolist()

## 10. Filter trips to the selected weekday

We keep only trips whose `service_id` is active on Tuesday, May 5, 2026.

In [ ]:
weekday_trips = trips.loc[
    trips['service_id'].isin(active_services)
].copy()

print(f'Total trips in full GTFS feed: {len(trips):,}')
print(f'Trips operating on {ANALYSIS_DATE:%Y-%m-%d}: {len(weekday_trips):,}')
print(f'Unique routes operating that day: {weekday_trips["route_id"].nunique()}')

weekday_trips.head()

## 11. Join trips with route names

The `trips` table knows which `route_id` each trip belongs to, while the `routes` table contains route numbers and readable names. We merge them so later analysis is easier to interpret.

In [ ]:
weekday_trips_with_routes = weekday_trips.merge(
    routes,
    on='route_id',
    how='left',
    validate='many_to_one'
)

weekday_trips_with_routes[[
    'route_id', 'route_short_name', 'route_long_name',
    'trip_id', 'direction_id', 'shape_id', 'service_id'
]].head()

## 12. Create a route-level summary for the selected weekday

This is our first clean analytical summary:
- one row per route,
- number of scheduled trips that day,
- number of route directions,
- number of distinct shapes used.

In [ ]:
weekday_route_summary = (
    weekday_trips_with_routes
    .groupby(['route_id', 'route_short_name', 'route_long_name'], as_index=False)
    .agg(
        scheduled_trips=('trip_id', 'nunique'),
        directions=('direction_id', 'nunique'),
        unique_shapes=('shape_id', 'nunique')
    )
    .sort_values(['scheduled_trips', 'route_short_name'], ascending=[False, True])
)

weekday_route_summary.head(20)

## 13. Quick sanity checks

Before moving forward, verify that:
- route IDs were successfully matched to route names,
- every active trip has a shape ID,
- the core tables look complete enough for later metrics.

In [ ]:
sanity_checks = {
    'weekday trips': len(weekday_trips_with_routes),
    'unique active routes': weekday_trips_with_routes['route_id'].nunique(),
    'missing route names': weekday_trips_with_routes['route_short_name'].isna().sum(),
    'missing shape IDs': weekday_trips_with_routes['shape_id'].isna().sum(),
    'unique stops in feed': stops['stop_id'].nunique(),
    'unique shapes in feed': shapes['shape_id'].nunique(),
}

pd.Series(sanity_checks, name='value').to_frame()

## 14. Export the cleaned starting tables

These CSVs are optional, but useful for keeping intermediate outputs organized.

In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

weekday_trips_with_routes.to_csv(OUTPUT_DIR / 'weekday_trips_with_routes_2026-05-05.csv', index=False)
weekday_route_summary.to_csv(OUTPUT_DIR / 'weekday_route_summary_2026-05-05.csv', index=False)

print('Exported:')
print('-', OUTPUT_DIR / 'weekday_trips_with_routes_2026-05-05.csv')
print('-', OUTPUT_DIR / 'weekday_route_summary_2026-05-05.csv')

# What this notebook accomplished

At this point, we have:

- loaded the MiWay GTFS feed,
- confirmed its date coverage,
- selected a normal weekday schedule,
- identified all trips operating that day,
- joined trips to human-readable route names,
- and created the first route-level summary table.

## Next notebook
Notebook 2 will calculate **route geometry metrics**:

1. route path length from `shapes.txt`,
2. start-to-end straight-line distance,
3. route directness ratio,
4. unique stop count,
5. stop density.